In [1]:
import os
import random
import numpy as np
import torch

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

torch.use_deterministic_algorithms(True, warn_only=True)

In [2]:
!uv pip install -q gdown
!gdown --id 1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS -O dataset.zip
!unzip -q dataset.zip

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS
From (redirected): https://drive.google.com/uc?id=1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS&confirm=t&uuid=6d5726a2-89fc-4b55-b722-6a9ec1ffddac
To: /kaggle/working/dataset.zip
100%|████████████████████████████████████████| 356M/356M [00:11<00:00, 31.0MB/s]


In [3]:
%pip install comet_ml -qq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 786.2/786.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 18.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [4]:
import os
import sys

# Keep Comet tracking enabled, but hide INFO-level console messages.
os.environ["COMET_LOGGING_CONSOLE"] = "ERROR"

sys.path.append('/kaggle/input/datasets/maksimbessolitsyn/yambdadataset')
sys.path.append('/kaggle/input/models/maksimbessolitsyn/sasrec/pytorch/default/35')

In [5]:
import logging
import warnings

import comet_ml
import torch

# Silence torch warnings/noisy logs, keep tqdm untouched.
warnings.filterwarnings("ignore", category=UserWarning, module=r"torch(\.|$)")
warnings.filterwarnings("ignore", category=FutureWarning, module=r"torch(\.|$)")
logging.getLogger("torch").setLevel(logging.ERROR)
logging.getLogger("torch._dynamo").setLevel(logging.ERROR)

# Suppress Comet INFO messages in notebook output.
logging.getLogger("comet_ml").setLevel(logging.ERROR)

from model import Graph, Tau
from train import train_loop
from eval import eval_loop
from dataset import (TrainingDataset, TestDataset, TrainingBatch, get_train_histories,
                     get_train_events, get_general_data, get_item_to_token,
                     get_test_histories, get_item_to_freq)

from config import VOCAB_SIZE, MAX_TRAIN_EVENTS_PER_USER, MAX_LEN, COMET_API_KEY

In [6]:
comet_ml.login(api_key=COMET_API_KEY)

In [7]:
train, test, embeddings, artists, test_targets = get_general_data()
item_to_token = get_item_to_token(train, vocab_size=None)
item_to_freq = get_item_to_freq(train)

train_events = get_train_events(train, item_to_token, item_to_freq=item_to_freq)
train_histories = get_train_histories(train_events)

test_histories = get_test_histories(test, train_events)

vocab_size = item_to_token['token_id'].max() + 1

In [8]:
dataloader = TrainingDataset(train_histories, batch_size=32,
                             seq_len=MAX_TRAIN_EVENTS_PER_USER,
                             shuffle=True, device='cuda',
                             uniform_negative_items=30_000,
                             in_batch_negative_items=1_000,
                             vocab_size=vocab_size)

total_num_tokens = dataloader.total_num_tokens

In [9]:
class ConstantTau(Tau):
    def __init__(self, tau: float = 0.05) -> None:
        super().__init__(tau_min=tau,
                         tau_max=tau,
                         num_epochs=0,
                         num_tokens_per_epoch=0)
        self.tau = float(tau)

    def __call__(self, net: 'Graph',
                 pos_logits: torch.Tensor,
                 neg_logits: torch.Tensor,
                 batch: TrainingBatch, *args, **kwargs) -> torch.Tensor:
        return torch.tensor(self.tau, device=pos_logits.device, dtype=torch.float32)

    def experiment_name(self) -> str:
        return f"Constant[value={self.tau:.5g}]"

In [10]:
from torch import nn

class ParameterTau(Tau):
    def __init__(self,
                 initial_tau: float = 0.3,
                 tau_min: float = 0.055,
                 tau_max: float = 0.06,
                 num_epochs: int = 5,
                 num_tokens_per_epoch: int = 4_019_032) -> None:
        super().__init__(tau_min=tau_min,
                         tau_max=tau_max,
                         num_epochs=num_epochs,
                         num_tokens_per_epoch=num_tokens_per_epoch)
        self.initial_tau = float(initial_tau)
        self.tau = nn.Parameter(torch.tensor(self.initial_tau, dtype=torch.float32))

    def __call__(self, net: 'Graph',
                 pos_logits: torch.Tensor,
                 neg_logits: torch.Tensor,
                 batch: TrainingBatch, *args, **kwargs) -> torch.Tensor:
        return self.tau

    def experiment_name(self) -> str:
        return f"Param[init={self.initial_tau:.5g}]"


In [11]:
class LinearTau(Tau):
    def __call__(self, net: 'Graph',
                 pos_logits: torch.Tensor,
                 neg_logits: torch.Tensor,
                 batch: TrainingBatch, *args, **kwargs) -> torch.Tensor:
        tokens_passed = net.tokens_passed.to(device=pos_logits.device, dtype=torch.float32)
        phase = tokens_passed / (float(self.num_tokens_per_epoch) * float(self.num_epochs))
        return self.tau_max + phase * (self.tau_min - self.tau_max)

    def experiment_name(self) -> str:
        return (
            f"Linear[min={float(self.tau_min):.5g},"
            f"max={float(self.tau_max):.5g},"
            f"epochs={int(self.num_epochs)}]"
        )


In [12]:
class CosTau(Tau):
    def __call__(self, net: 'Graph',
                 pos_logits: torch.Tensor,
                 neg_logits: torch.Tensor,
                 batch: TrainingBatch, *args, **kwargs) -> torch.Tensor:
        tokens_passed = net.tokens_passed.to(device=pos_logits.device, dtype=torch.float32)
        epoch = tokens_passed / float(self.num_tokens_per_epoch)
        phase = epoch / float(self.num_epochs)
        return self.tau_min + torch.cos(phase * (torch.pi / 2)) * (self.tau_max - self.tau_min)

    def experiment_name(self) -> str:
        return (
            f"Cos[min={float(self.tau_min):.5g},"
            f"max={float(self.tau_max):.5g},"
            f"epochs={int(self.num_epochs)}]"
        )


In [13]:
def run_experiment(tau: Tau, num_epochs=5):
    experiment = comet_ml.Experiment(
        api_key=COMET_API_KEY,
        project_name="AdaptiveTemperature",
        workspace="maksim-bessolitsyn",
    )
    experiment.set_name(tau.experiment_name() + f" Epochs:{num_epochs}")
    
    graph = Graph(
        vocab_size=vocab_size,
        max_seq_len=MAX_LEN,
        n_layers=4,
        dropout=0.1,
        tau=tau,
        log_q_correction=1,
        is_cosine_similarity=True,
    ).cuda()
    compiled_graph = torch.compile(graph, mode='default')
    
    optimizer = torch.optim.AdamW(
        compiled_graph.parameters(),
        lr=1e-3,
        weight_decay=1e-4,
        fused=True
    )
    
    train_loop(
        compiled_graph,
        num_epochs=num_epochs,
        train_dataloader=dataloader,
        optimizer=optimizer,
        grad_clip=10.0,
        grad_accum_steps=1,
        writer=experiment,
    )
    
    result = eval_loop(
        compiled_graph,
        TestDataset(test_histories, batch_size=128),
        test_histories=test_histories,
        test_targets=test_targets,
        item_to_token=item_to_token,
        writer=experiment,
    )
    
    experiment.end()
    
    return result

In [14]:
for tau in [0.04, 0.045, 0.05, 0.055]:
    print(
        run_experiment(ConstantTau(tau=tau), num_epochs=15),
        tau,
    )

100%|██████████| 293/293 [00:10<00:00, 28.06it/s]


{'hitrate': 0.3410778187256316, 'recall': 0.11395944209807399, 'ndcg': 0.0452254604649848, 'coverage': 0.3104093358870429} 0.04


100%|██████████| 293/293 [00:10<00:00, 28.08it/s]


{'hitrate': 0.3512524702237889, 'recall': 0.11949002806893694, 'ndcg': 0.049056676368799776, 'coverage': 0.3004384150880966} 0.045


100%|██████████| 293/293 [00:10<00:00, 28.02it/s]


{'hitrate': 0.3466591892324948, 'recall': 0.1180767160146886, 'ndcg': 0.04742776990194541, 'coverage': 0.25817494607303526} 0.05


100%|██████████| 293/293 [00:10<00:00, 28.12it/s]


{'hitrate': 0.34075735726112266, 'recall': 0.1160526750023391, 'ndcg': 0.04637995821432303, 'coverage': 0.24819129914671315} 0.055


In [15]:
print(
    run_experiment(ParameterTau(initial_tau=0.05), num_epochs=15), 
    tau,
)

epoch 0:   0%|          | 0/1255 [00:00<?, ?it/s]/kaggle/input/models/maksimbessolitsyn/sasrec/pytorch/default/35/model/model/graph.py:74: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  writer.log_metric("train/tau", value=float(tau.mean()), step=step) if writer is not None else None
100%|██████████| 293/293 [00:10<00:00, 28.20it/s]


{'hitrate': 0.2800299097366875, 'recall': 0.086511709784989, 'ndcg': 0.03249077469759841, 'coverage': 0.5406186170517381} 0.055


In [16]:
cases = [
    (0.035, 0.055),
    (0.04, 0.055),
    (0.04, 0.06),
]

In [17]:
for tau_min, tau_max in cases:
    print(
        run_experiment(
            LinearTau(tau_min=tau_min,
                      tau_max=tau_max,
                      num_epochs=15,
                     ),
            num_epochs=15
        ),
        tau_min, 
        tau_max, 
    )

100%|██████████| 293/293 [00:10<00:00, 27.92it/s]


{'hitrate': 0.3397692677455536, 'recall': 0.11381299216299962, 'ndcg': 0.04594093240308334, 'coverage': 0.32136016849392646} 0.035 0.055


100%|██████████| 293/293 [00:10<00:00, 28.10it/s]


{'hitrate': 0.34342786946536347, 'recall': 0.11501733899875934, 'ndcg': 0.04593255361812432, 'coverage': 0.30545887233785324} 0.04 0.055


100%|██████████| 293/293 [00:10<00:00, 28.16it/s]


{'hitrate': 0.3453506382524168, 'recall': 0.11604467790558083, 'ndcg': 0.04634533857284265, 'coverage': 0.28573973796903734} 0.04 0.06


In [18]:
for tau_min, tau_max in cases:
    print(
        run_experiment(
            CosTau(tau_min=tau_min,
                   tau_max=tau_max,
                   num_epochs=15,
                  ),
            num_epochs=15
        ),
        tau_min, 
        tau_max, 
    )

100%|██████████| 293/293 [00:10<00:00, 27.94it/s]


{'hitrate': 0.3311168082038135, 'recall': 0.10939373648669035, 'ndcg': 0.04318136733914769, 'coverage': 0.3578141603619311} 0.035 0.055


100%|██████████| 293/293 [00:10<00:00, 28.11it/s]


{'hitrate': 0.34708647118517333, 'recall': 0.11795546164763539, 'ndcg': 0.04714180459175646, 'coverage': 0.3407356974235955} 0.04 0.055


100%|██████████| 293/293 [00:10<00:00, 27.91it/s]


{'hitrate': 0.33581690968327726, 'recall': 0.11121564910118957, 'ndcg': 0.04337206163011018, 'coverage': 0.29287909542686613} 0.04 0.06
